# 환경 점검 (setup_check)

> **실습 44개를 실행하기 전에 이 파일을 한 번 돌려 보세요.**

---

## 이 장이 하는 일

실습에 필요한 것들이 제대로 준비됐는지 **한 번에 확인**합니다.
문제가 있으면 무엇을 어떻게 고쳐야 하는지 함께 알려줍니다.

| 절 | 확인 내용 |
|---|---|
| 1 | Python 버전과 가상환경 |
| 2 | 필수 패키지 |
| 3 | GPU와 PyTorch |
| 4 | 실습 규모 판정 |
| 5 | 폴더 구조와 설정 파일 |
| 6 | 실제 연산 시험 |
| 7 | 종합 판정 |

**모든 셀을 순서대로 실행**하고 마지막 요약을 확인하세요.

> 설치를 아직 하지 않았다면 먼저 `bash setup.sh` 를 실행하거나
> 1장의 절차를 따르세요.

---

## 1. Python 버전과 가상환경

In [ ]:
import sys
import platform
import os
from pathlib import Path

results = {}      # 점검 결과를 모아 7절에서 요약한다

print("=" * 70)
print("1. Python 환경")
print("=" * 70)

version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
print(f"  Python 버전  : {version}")
print(f"  실행 파일    : {sys.executable}")
print(f"  운영체제     : {platform.system()} {platform.release()}")
print(f"  아키텍처     : {platform.machine()}")
print()

# 버전 판정
major, minor = sys.version_info.major, sys.version_info.minor
if major == 3 and 10 <= minor <= 13:
    print("  [OK] 지원 범위입니다 (3.10~3.13, 권장 3.13)")
    results["python"] = True
elif major == 3 and minor >= 14:
    print("  [주의] 3.14 이상입니다.")
    print("         일부 패키지의 휠이 아직 없을 수 있습니다.")
    print("         문제가 생기면 3.13 을 권합니다. (1장 1-2절)")
    results["python"] = True
else:
    print("  [문제] Python 3.10 이상이 필요합니다.")
    print("         https://www.python.org/downloads/windows/")
    results["python"] = False

print()

# 가상환경 확인
in_venv = (hasattr(sys, "real_prefix") or
           (hasattr(sys, "base_prefix") and sys.base_prefix != sys.prefix))
venv_path = os.environ.get("VIRTUAL_ENV")

if in_venv or venv_path:
    print(f"  [OK] 가상환경 사용 중")
    if venv_path:
        print(f"       {venv_path}")
    results["venv"] = True
else:
    print("  [주의] 가상환경이 아닙니다.")
    print("         전역에 설치하면 다른 프로젝트와 충돌할 수 있습니다.")
    print("         1장 1-3절을 참조해 가상환경을 만드는 편이 좋습니다.")
    results["venv"] = False

print()

# ── 터미널 한글 출력 확인 (Windows에서 자주 문제가 된다) ──
print("  한글 출력 시험 : 가나다 ABC 123")

enc = (sys.stdout.encoding or "").lower()
print(f"  출력 인코딩    : {sys.stdout.encoding}")

if platform.system() == "Windows" and "utf" not in enc:
    print()
    print("  [주의] 위 '가나다'가 깨져 보인다면 콘솔 인코딩 문제입니다.")
    print("         노트북 안에서는 대개 정상이지만, PowerShell 터미널에서")
    print("         git log 나 python 출력을 볼 때 깨질 수 있습니다.")
    print()
    print("         해결 (PowerShell 에서 한 번만):")
    print("           New-Item -Path $PROFILE -Type File -Force")
    print("           notepad $PROFILE")
    print("         열린 파일에 아래 두 줄을 넣고 저장합니다.")
    print("           [Console]::OutputEncoding = [System.Text.Encoding]::UTF8")
    print("           $OutputEncoding = [System.Text.Encoding]::UTF8")
    print()
    print("         자세한 내용은 1장 1-7절 참조")
    results["encoding"] = None      # 판정 대상은 아니고 안내만
else:
    print("  [OK] 인코딩 설정에 문제가 없어 보입니다.")
    results["encoding"] = True

---

## 2. 필수 패키지

노트북별로 필요한 패키지를 확인합니다.
**선택 항목은 없어도 대부분의 실습이 가능합니다.**

In [ ]:
import importlib

print("=" * 78)
print("2. 패키지 점검")
print("=" * 78)

package_groups = {
    "핵심 (전 노트북)": [
        ("numpy", "수치 계산", True),
        ("matplotlib", "시각화", True),
        ("pandas", "표 데이터", False),
    ],
    "머신러닝 (07~11장)": [
        ("sklearn", "scikit-learn", True),
        ("torch", "PyTorch", True),
    ],
    "딥러닝 (12~22장)": [
        ("torchvision", "이미지 데이터셋", True),
        ("gymnasium", "강화학습 (17장)", False),
    ],
    "LLM (23~25장)": [
        ("transformers", "Hugging Face", True),
        ("openai", "API 연동 (25장)", False),
        ("dotenv", "환경 변수 (25장)", False),
    ],
    "검색·RAG (26~29장)": [
        ("sentence_transformers", "문장 임베딩", True),
        ("chromadb", "벡터 DB (28장)", False),
    ],
    "파인튜닝 (30~34장)": [
        ("peft", "LoRA (32장)", False),
        ("accelerate", "학습 보조", False),
    ],
    "기타": [
        ("PIL", "이미지 (36장)", False),
        ("tqdm", "진행 표시", False),
        ("fastapi", "API 서버 (40장)", False),
    ],
}

installed, missing_required, missing_optional = [], [], []

for group, items in package_groups.items():
    print(f"\n[{group}]")
    for name, desc, required in items:
        try:
            mod = importlib.import_module(name)
            ver = getattr(mod, "__version__", "설치됨")
            print(f"  [OK]   {name:<24}{str(ver):<14}{desc}")
            installed.append(name)
        except ImportError:
            mark = "필수" if required else "선택"
            print(f"  [{mark}] {name:<24}{'':<14}{desc}")
            (missing_required if required else missing_optional).append(name)

print()
print("-" * 78)
print(f"설치됨 {len(installed)}개 / 필수 누락 {len(missing_required)}개 / "
      f"선택 누락 {len(missing_optional)}개")

results["packages"] = len(missing_required) == 0

if missing_required:
    print()
    print("  [문제] 필수 패키지가 없습니다:")
    print(f"         {', '.join(missing_required)}")
    print()
    print("  해결: pip install -r requirements.txt")
    print("        (torch 는 별도 — 원고 1.6절 참조)")

if missing_optional:
    print()
    print("  선택 패키지 미설치 — 해당 장에서만 필요합니다:")
    for name in missing_optional:
        print(f"    pip install {name}")

---

## 3. GPU와 PyTorch

**GPU가 없어도 대부분의 실습(1~31장)이 가능합니다.**
다만 있으면 훨씬 빠릅니다.

In [ ]:
print("=" * 70)
print("3. PyTorch 와 GPU")
print("=" * 70)

try:
    import torch

    print(f"  PyTorch 버전 : {torch.__version__}")
    cuda_available = torch.cuda.is_available()
    print(f"  CUDA 사용 가능: {cuda_available}")
    print()

    if cuda_available:
        print(f"  [OK] GPU 를 사용할 수 있습니다")
        print(f"       이름     : {torch.cuda.get_device_name(0)}")
        props = torch.cuda.get_device_properties(0)
        vram = props.total_memory / 1024**3
        print(f"       VRAM     : {vram:.1f} GB")
        print(f"       CUDA     : {torch.version.cuda}")
        print(f"       Compute  : {props.major}.{props.minor}")
        results["gpu"] = True
        results["vram"] = vram
    else:
        results["gpu"] = False
        results["vram"] = 0

        if "+cpu" in torch.__version__:
            print("  [주의] CPU 전용 PyTorch 가 설치되어 있습니다.")
            print()
            print("  GPU 가 있는데도 이렇게 나온다면 재설치가 필요합니다:")
            print("    pip uninstall torch torchvision torchaudio")
            print("    pip install torch torchvision torchaudio \\")
            print("        --index-url https://download.pytorch.org/whl/cu128")
            print()
            print("  (원고 1.10절 '자주 겪는 문제' 참조)")
        else:
            print("  [정보] GPU 를 찾지 못했습니다.")
            print()
            print("  확인할 것:")
            print("    1) NVIDIA GPU 가 있는가")
            print("    2) 드라이버가 설치됐는가 (nvidia-smi 로 확인)")
            print("    3) 설치 후 재부팅했는가")
            print()
            print("  GPU 가 없어도 1~31장은 CPU 로 실행 가능합니다.")

    results["torch"] = True

except ImportError:
    print("  [문제] PyTorch 가 설치되지 않았습니다.")
    print()
    print("  설치: pip install torch torchvision torchaudio \\")
    print("            --index-url https://download.pytorch.org/whl/cu128")
    print("        (GPU 없으면 cu128 대신 cpu)")
    results["torch"] = False
    results["gpu"] = False
    results["vram"] = 0

---

## 4. 실습 규모 판정

GPU 메모리에 따라 **어느 실습까지 가능한지** 판정합니다.
1권 22.5절의 계산 기준을 따릅니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform as _platform
import numpy as np

# 한글 폰트
_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
_font = None
for _n in _c.get(_platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        _font = _n
        break
plt.rcParams["axes.unicode_minus"] = False
results["font"] = _font is not None

vram = results.get("vram", 0)

print("=" * 78)
print("4. 실습 규모 판정")
print("=" * 78)
print(f"  기준: VRAM {vram:.1f} GB" if vram > 0 else "  기준: GPU 없음 (CPU)")
print()

usable = vram * 0.85 if vram > 0 else 0

scenarios = [
    ("1~11장 (기초·신경망)",    0.0,  "CPU 로 충분"),
    ("12~22장 (CNN·Transformer)", 0.0,  "CPU 가능 (GPU 면 빠름)"),
    ("23~29장 (LLM·RAG)",      0.0,  "CPU 가능"),
    ("30~34장 (파인튜닝, 소형)",  2.0,  "DistilGPT-2 급"),
    ("32장 QLoRA (3B급)",      5.0,  "양자화 필요"),
    ("32장 QLoRA (7B급)",      6.5,  "양자화 + 짧은 시퀀스"),
    ("추론 INT4 (7B급)",       4.5,  "양자화"),
    ("전체 파인튜닝 (1B급)",     9.0,  "가중치 x4"),
]

print(f"{'실습':<30}{'필요 VRAM':<14}{'판정':<14}{'비고'}")
print("-" * 78)
for name, need, note in scenarios:
    if need == 0.0:
        verdict = "가능"
    elif vram == 0:
        verdict = "GPU 필요"
    elif usable >= need:
        verdict = "가능"
    else:
        verdict = "어려움"
    need_str = "CPU" if need == 0 else f"{need:.1f} GB"
    print(f"{name:<30}{need_str:<14}{verdict:<14}{note}")
print("-" * 78)
print()

if vram == 0:
    print("  GPU 가 없어도 **1~31장 전부** 실행 가능합니다.")
    print("  일부 학습이 느릴 뿐이며, 각 실습은 CPU 기준으로 규모를 조정했습니다.")
elif vram >= 8:
    print("  8GB 이상 — 이 책의 모든 실습을 진행할 수 있습니다.")
elif vram >= 6:
    print("  6GB 대 — 대부분 가능하나 7B 급 QLoRA 는 배치를 줄여야 합니다.")
else:
    print("  4GB 대 — 파인튜닝 실습은 더 작은 모델로 대체하세요.")

# 시각화
fig, ax = plt.subplots(figsize=(9, 4))
names = [s[0].split(" (")[0] for s in scenarios]
needs = [s[1] for s in scenarios]
colors = ["#0D9488" if (n == 0 or usable >= n) else "#DC2626" for n in needs]

bars = ax.barh(range(len(names)), [max(n, 0.3) for n in needs], color=colors)
if vram > 0:
    ax.axvline(usable, color="#1E40AF", linestyle="--", linewidth=2)
    ax.text(usable + 0.15, len(names) - 0.5,
            f"사용 가능\n{usable:.1f}GB", fontsize=8, color="#1E40AF")
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel("필요 VRAM (GB)  —  회색 막대는 CPU 가능")
ax.set_title(f"실습 가능 범위 (현재 VRAM {vram:.1f}GB)" if vram > 0
             else "실습 가능 범위 (GPU 없음 — 초록은 CPU 가능)")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

---

## 5. 폴더 구조와 설정 파일

In [ ]:
from pathlib import Path

print("=" * 78)
print("5. 저장소 구조")
print("=" * 78)

root = Path.cwd()
print(f"  현재 위치: {root}")
print()

expected = {
    "part1": ("dir", "1부 노트북", True),
    "part2": ("dir", "2부 노트북", True),
    "part3": ("dir", "3부 노트북", True),
    "part4": ("dir", "4부 노트북", True),
    "part5": ("dir", "5부 노트북", True),
    "part6": ("dir", "6부 노트북", True),
    "part7": ("dir", "7부 노트북", True),
    "part8": ("dir", "8부 종합 프로젝트", True),
    "utils": ("dir", "공통 함수 (42장에서 생성)", False),
    "data": ("dir", "데이터셋 저장", False),
    "requirements.txt": ("file", "패키지 목록", True),
    ".env.example": ("file", "API 키 양식", True),
    ".env": ("file", "실제 API 키 (21장~)", False),
    ".gitignore": ("file", "Git 제외 목록", False),
    "README.md": ("file", "안내 문서", False),
}

print(f"{'항목':<24}{'종류':<10}{'상태':<12}{'설명'}")
print("-" * 78)

missing_essential = []
for name, (kind, desc, essential) in expected.items():
    path = root / name
    exists = path.is_dir() if kind == "dir" else path.is_file()

    if exists:
        if kind == "dir" and name.startswith("part"):
            n = len(list(path.glob("*.ipynb")))
            status = f"OK ({n}개)"
        else:
            status = "OK"
    else:
        status = "없음"
        if essential:
            missing_essential.append(name)

    print(f"{name:<24}{kind:<10}{status:<12}{desc}")

print("-" * 78)

results["structure"] = len(missing_essential) == 0

if missing_essential:
    print()
    print(f"  [문제] 필요한 항목이 없습니다: {', '.join(missing_essential)}")
    print("         저장소 루트에서 이 실습을 실행했는지 확인하세요.")
else:
    print()
    print("  [OK] 구조가 정상입니다.")

# 노트북 개수 확인
total_nb = sum(len(list((root / p).glob("*.ipynb")))
               for p in expected if p.startswith("part") and (root / p).is_dir())
print(f"  전체 실습: {total_nb}개")

# .env 안내
if not (root / ".env").exists() and (root / ".env.example").exists():
    print()
    print("  [안내] .env 가 없습니다. API 실습(21장~)을 하려면 만드세요:")
    print("         cp .env.example .env")
    print("         (Windows: copy .env.example .env)")

---

## 6. 실제 연산 시험

**설치만 됐다고 되는 것이 아닙니다.** 실제로 계산이 되는지 확인합니다.

In [ ]:
import time
import numpy as np

print("=" * 70)
print("6. 연산 시험")
print("=" * 70)

test_results = {}

# ── NumPy ──
print("\n[NumPy]")
try:
    A = np.array([[2.0, -1.0], [1.0, 3.0]])
    x = np.array([3.0, 2.0])
    y = A @ x
    ok = np.allclose(y, [4.0, 9.0])
    print(f"  A @ x = {y}  (1권 4.2절 값: [4, 9])")
    print(f"  [{'OK' if ok else '실패'}]")
    test_results["numpy"] = ok
except Exception as e:
    print(f"  [실패] {e}")
    test_results["numpy"] = False

# ── PyTorch CPU ──
print("\n[PyTorch — CPU]")
try:
    import torch
    a = torch.randn(500, 500)
    b = torch.randn(500, 500)
    t0 = time.time()
    c = a @ b
    cpu_time = time.time() - t0
    print(f"  500x500 행렬 곱: {cpu_time*1000:.1f} ms")
    print(f"  [OK]")
    test_results["torch_cpu"] = True
except Exception as e:
    print(f"  [실패] {e}")
    test_results["torch_cpu"] = False

# ── autograd ──
print("\n[PyTorch — 자동 미분]")
try:
    import torch
    # 1권 10.4절 값 검증 (12장에서 다루는 것)
    x_t = torch.tensor([1.0, 0.5])
    W1 = torch.tensor([[0.2, 0.4], [0.1, 0.3]], requires_grad=True)
    W2 = torch.tensor([0.6, 0.9], requires_grad=True)
    a1 = torch.sigmoid(W1 @ x_t)
    z2 = W2 @ a1
    loss = (z2 - 1.0) ** 2
    loss.backward()

    expected = np.array([-0.1614, -0.1516])
    ok = np.allclose(W2.grad.numpy(), expected, atol=1e-3)
    print(f"  dL/dW2 = {W2.grad.numpy().round(4)}")
    print(f"  1권 10.4절 값: [-0.1614, -0.1516]")
    print(f"  [{'OK' if ok else '실패'}]")
    test_results["autograd"] = ok
except Exception as e:
    print(f"  [실패] {e}")
    test_results["autograd"] = False

# ── GPU ──
print("\n[PyTorch — GPU]")
try:
    import torch
    if torch.cuda.is_available():
        a_g = a.cuda()
        b_g = b.cuda()
        _ = a_g @ b_g
        torch.cuda.synchronize()

        t0 = time.time()
        _ = a_g @ b_g
        torch.cuda.synchronize()
        gpu_time = time.time() - t0

        print(f"  500x500 행렬 곱: {gpu_time*1000:.1f} ms")
        print(f"  CPU 대비: {cpu_time/gpu_time:.1f}배")
        print(f"  [OK]")
        test_results["torch_gpu"] = True
    else:
        print("  건너뜀 (GPU 없음)")
        test_results["torch_gpu"] = None
except Exception as e:
    print(f"  [실패] {e}")
    test_results["torch_gpu"] = False

# ── 한글 폰트 ──
print("\n[matplotlib 한글]")
if results.get("font"):
    print(f"  폰트: {plt.rcParams['font.family']}")
    print("  [OK]")
    test_results["font"] = True
else:
    print("  [주의] 한글 폰트를 찾지 못했습니다.")
    print("         그래프의 한글이 네모로 보일 수 있습니다.")
    print("         Windows 는 '맑은 고딕'이 기본 설치되어 있어야 합니다.")
    test_results["font"] = False

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("그래프 출력 시험")
print("=" * 70)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# 한글 표시 확인
ax = axes[0]
x = np.linspace(0, 10, 100)
ax.plot(x, np.sin(x), linewidth=2, color="#1E40AF", label="사인 함수")
ax.plot(x, np.cos(x), linewidth=2, color="#EA580C",
        linestyle="--", label="코사인 함수")
ax.set_xlabel("가로축 (한글 확인)")
ax.set_ylabel("세로축")
ax.set_title("한글이 제대로 보이나요?")
ax.legend()
ax.grid(alpha=0.3)

# 음수 부호 확인
ax = axes[1]
values = [-3, -1, 0, 2, 4]
ax.bar(range(len(values)), values,
       color=["#DC2626" if v < 0 else "#0D9488" for v in values])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(range(len(values)))
ax.set_xticklabels([str(v) for v in values])
ax.set_title("음수 부호가 깨지지 않나요?")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("위 그래프에서 확인할 것")
print("  1) 한글이 네모(□)로 보이지 않는가")
print("  2) 음수 부호(-3, -1)가 제대로 표시되는가")
print()
print("문제가 있다면 3장 1절 '한글 폰트 설정'을 참조하세요.")

---

## 7. 종합 판정

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("=" * 78)
print("종합 판정")
print("=" * 78)
print()

checks = [
    ("Python 버전",      results.get("python", False),  True),
    ("가상환경",         results.get("venv", False),    False),
    ("필수 패키지",      results.get("packages", False), True),
    ("PyTorch",         results.get("torch", False),   True),
    ("저장소 구조",      results.get("structure", False), True),
    ("NumPy 연산",      test_results.get("numpy", False), True),
    ("PyTorch 연산",    test_results.get("torch_cpu", False), True),
    ("자동 미분",        test_results.get("autograd", False), True),
    ("한글 폰트",        test_results.get("font", False), False),
    ("GPU",             results.get("gpu", False),     False),
]

print(f"{'항목':<24}{'결과':<14}{'필수 여부'}")
print("-" * 78)
for name, ok, required in checks:
    mark = "통과" if ok else ("실패" if required else "선택 — 없음")
    req = "필수" if required else "선택"
    print(f"{name:<24}{mark:<14}{req}")
print("-" * 78)
print()

essential_ok = all(ok for _, ok, req in checks if req)
optional_count = sum(1 for _, ok, req in checks if not req and ok)
optional_total = sum(1 for _, _, req in checks if not req)

if essential_ok:
    print("  [준비 완료] 필수 항목을 모두 통과했습니다.")
    print(f"              선택 항목: {optional_count}/{optional_total}")
    print()
    print("  다음 단계")
    print("    part1/01_setup_environment.ipynb 부터 시작하세요.")
    print()
    if not results.get("gpu"):
        print("  GPU 가 없어도 1~31장은 CPU 로 실행됩니다.")
        print("  24~26장 파인튜닝은 작은 모델을 쓰도록 구성했습니다.")
else:
    failed = [name for name, ok, req in checks if req and not ok]
    print("  [준비 미완료] 아래 항목을 해결해야 합니다:")
    for name in failed:
        print(f"    - {name}")
    print()
    print("  해결 방법")
    print("    1) bash setup.sh 실행")
    print("    2) 또는 1장의 절차를 따르기")
    print("    3) 위 각 절의 안내 메시지 확인")

# 시각화
fig, ax = plt.subplots(figsize=(8.5, 4))
names = [c[0] for c in checks]
values = [1 if c[1] else 0 for c in checks]
colors = []
for name, ok, req in checks:
    if ok:
        colors.append("#0D9488")
    elif req:
        colors.append("#DC2626")
    else:
        colors.append("#94A3B8")

ax.barh(range(len(names)), values, color=colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xticks([0, 1])
ax.set_xticklabels(["미통과", "통과"])
ax.set_title("환경 점검 결과  (초록=통과, 빨강=필수 실패, 회색=선택)")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 78)
print("문제 해결 참조표")
print("=" * 78)
print()
print(f"{'증상':<34}{'원인 / 해결'}")
print("-" * 78)
troubles = [
    ("python 명령을 찾을 수 없음",     "PATH 미등록 — 원고 1.4절 재설치"),
    ("torch.cuda.is_available() False", "CPU 버전 설치됨 — 원고 1.10절"),
    ("ModuleNotFoundError",           "pip install -r requirements.txt"),
    ("VS Code 에서 import 실패",       "인터프리터를 venv 로 지정 (Ctrl+Shift+P)"),
    ("그래프 한글이 네모",              "3장 1절 폰트 설정"),
    ("CUDA out of memory",            "배치 크기 축소 — 원고 25장 9절"),
    ("모델 다운로드 실패",              "네트워크 확인, 캐시 폴더 권한 확인"),
    ("커널이 자꾸 죽음",               "메모리 부족 — 다른 프로그램 종료"),
]
for a, b in troubles:
    print(f"{a:<34}{b}")
print("-" * 78)
print()
print("더 자세한 내용은 1장 '자주 겪는 문제와 해결' 을 참조하세요.")
print()
print("=" * 78)
print("학습 경로 안내")
print("=" * 78)
print()
print(f"{'Part':<10}{'노트북':<14}{'주제':<30}{'GPU'}")
print("-" * 78)
paths = [
    ("1부", "01~06", "환경·NumPy·시각화·선형회귀·전처리", "불필요"),
    ("2부", "07~11", "머신러닝·결정트리·퍼셉트론·역전파", "불필요"),
    ("3부", "12~15", "PyTorch·최적화·정규화·CNN", "권장"),
    ("4부", "16~19", "RNN·강화학습·생성 모델", "권장"),
    ("5부", "20~25", "Attention·Transformer·LLM 기초", "불필요"),
    ("6부", "26~33", "토크나이저·검색·RAG·파인튜닝", "일부 권장"),
    ("7부", "34~41", "정렬·추론·Agent·평가·배포·운영", "일부 권장"),
    ("8부", "42~44", "복습·종합 프로젝트 2종", "불필요"),
]
for a, b, c, d in paths:
    print(f"{a:<10}{b:<14}{c:<30}{d}")
print("-" * 78)
print()
print("순서대로 진행하는 것을 권합니다.")
print("각 실습이 앞의 내용을 전제로 하기 때문입니다.")